# Análise Exploratória

Antes de treinar qualquer modelo, preciso entender o que diferencia um hit de uma música que não estourou. Se não houver diferença clara nas features de áudio, não adianta seguir com machine learning — o sinal simplesmente não está ali.

In [ ]:
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

df = pd.read_csv("../data/processed/tracks_clean.csv")

FEATURES = [
    "danceability", "energy", "valence", "tempo",
    "loudness", "speechiness", "acousticness",
    "instrumentalness", "duration_ms",
]

df["is_hit_label"] = df["is_hit"].map({0: "Não-hit", 1: "Hit"})

print(f"Dataset: {df.shape[0]} músicas, {df.shape[1]} colunas")
print(f"Hits: {df['is_hit'].sum()} | Não-hits: {(df['is_hit'] == 0).sum()}")

## Estatísticas descritivas por classe

Primeiro passo: olhar as médias e desvios separados. Se a média de energy dos hits for praticamente igual à dos não-hits, essa feature provavelmente não vai ajudar o modelo.

In [ ]:
desc = df.groupby("is_hit")[FEATURES].agg(["mean", "median", "std"])
desc

## Distribuição de cada feature por classe

Os violin plots mostram não só a média, mas o formato da distribuição. Uma feature pode ter média parecida nas duas classes e ainda assim ter distribuições bem diferentes — é isso que quero checar.

In [ ]:
fig = make_subplots(rows=3, cols=3, subplot_titles=FEATURES)

for i, feat in enumerate(FEATURES):
    row, col = i // 3 + 1, i % 3 + 1
    for label, color in [("Não-hit", "#636EFA"), ("Hit", "#EF553B")]:
        subset = df[df["is_hit_label"] == label]
        fig.add_trace(
            go.Violin(
                y=subset[feat], name=label,
                legendgroup=label, scalegroup=feat,
                marker_color=color, showlegend=(i == 0),
            ),
            row=row, col=col,
        )

fig.update_layout(
    height=900, width=900,
    title_text="Features de áudio: Hit vs Não-hit",
    template="plotly_white",
)
fig.show()

## Correlação entre features

Se duas features forem muito correlacionadas (ex: energy e loudness costumam andar juntas), o modelo pode não precisar das duas. Também ajuda a entender o perfil geral das músicas.

In [ ]:
corr = df[FEATURES].corr()

fig = px.imshow(
    corr, text_auto=".2f",
    color_continuous_scale="RdBu_r",
    title="Correlação entre features de áudio",
    zmin=-1, zmax=1,
)
fig.update_layout(height=600, width=650, template="plotly_white")
fig.show()

## Features com maior diferença entre classes

Calculo a diferença absoluta entre as médias de cada classe e pego as 4 maiores. São as features que mais separam hits de não-hits, pelo menos na média.

In [ ]:
means = df.groupby("is_hit")[FEATURES].mean()
diff = (means.loc[1] - means.loc[0]).abs().sort_values(ascending=False)
top4 = diff.head(4).index.tolist()

print("Top 4 features com maior diferença entre classes:")
for feat in top4:
    print(f"  {feat}: hit={means.loc[1, feat]:.4f}  não-hit={means.loc[0, feat]:.4f}  diff={diff[feat]:.4f}")

In [ ]:
fig = make_subplots(rows=2, cols=2, subplot_titles=top4)

for i, feat in enumerate(top4):
    row, col = i // 2 + 1, i % 2 + 1
    for label, color in [("Não-hit", "#636EFA"), ("Hit", "#EF553B")]:
        subset = df[df["is_hit_label"] == label]
        fig.add_trace(
            go.Box(
                y=subset[feat], name=label,
                marker_color=color, showlegend=(i == 0),
                legendgroup=label,
            ),
            row=row, col=col,
        )

fig.update_layout(
    height=600, width=800,
    title_text="Box plots das 4 features com maior separação",
    template="plotly_white",
)
fig.show()

## Distribuição do target

Proporção de hits vs não-hits no dataset. Um desbalanceamento muito grande pode atrapalhar o modelo — vale saber o tamanho do problema antes de seguir.

In [ ]:
counts = df["is_hit_label"].value_counts().reset_index()
counts.columns = ["classe", "total"]

fig = px.bar(
    counts, x="classe", y="total", color="classe",
    color_discrete_map={"Hit": "#EF553B", "Não-hit": "#636EFA"},
    text="total",
    title="Distribuição do target: Hit vs Não-hit",
)
fig.update_layout(height=400, template="plotly_white", showlegend=False)
fig.show()

## Conclusões

**Músicas de sucesso têm energia maior?**

Em geral, sim. Hits tendem a ter energy mais alta que o restante. Faz sentido: músicas com mais energia costumam funcionar melhor em playlists de academia, festas e no rádio. Não quer dizer que toda música calma fracassa, mas a tendência está lá.

**Valência (positividade) importa?**

Menos do que eu esperava. A diferença entre hits e não-hits em valência é pequena. Músicas tristes estouram tanto quanto as alegres. Isso faz sentido com a popularidade de artistas como Billie Eilish e Olivia Rodrigo, que dominaram charts com músicas melancólicas.

**O BPM médio difere entre hits e não-hits?**

Quase nada. Hits acontecem em qualquer andamento. O tempo (BPM) sozinho não separa bem as classes — provavelmente porque existem hits lentos e rápidos em proporções parecidas.